# **Sesión 2:** Calibración de Cámara 📷⚙️

## 0. Preparación

In [68]:
import cv2
print("OpenCV should be 4.8.x Current version:", cv2.__version__)
from typing import List
import numpy as np
# import imageio --- IGNORE IF DEPRECATED---
import imageio.v3 as imageio
import copy
import os
import glob

OpenCV should be 4.8.x Current version: 4.8.0


## **Apartado A: Calibración de cámara** (derecha e izquierda)

En este apartado se realiza la calibración de dos cámaras de un sistema estereoscópico. Para ello se trabajará con las imágenes de las carpetas ``left`` y ``right``. En primer lugar se trabajará con la carpeta ``left``. Posteriormente, deberá repetir el proceso con las imágenes en la carpeta ``right``. Ambas carpetas contienen imágenes con las que se calibrarán las cámaras. En ellas aparece el patrón de calibración en diferentes posiciones y orientaciones. Estas imágenes serán los datos de entrada.

Los pasos que deberá seguir para calibrar una cámara son:

1. Defina y ejecute el método para cargar imágenes ``load_images()``.
2. Detecte las esquinas de los patrones usando ``cv2.findChessboardCorners()``. Refine las detecciones con ``cv2.cornerSubPix()``.
3. Compruebe que las detecciones son correctas dibujando los resultados con ``cv2.drawChessboardCorners()``.
4. Defina y ejecute el método ``get_chessboard_points(chessboard_shape, dx, dy)`` que proporcione las coordenadas 3D de las esquinas del patrón. El sistema de referencia utilizado deberá estar anclado al propio patrón.
5. Utilice ``cv2.calibrateCamera`` para obtener los parámetros de calibración para la cámara izquierda.

### **Tarea A.1:** Defina y ejecute el método para cargar imágenes ``load_images()``.

In [69]:
def load_images(filenames: List) -> List:
    return [imageio.imread(filename) for filename in filenames]

In [70]:
# TODO Build a list containing the paths of all images from the left camera
imgs_path = [filename for filename in glob.glob(os.path.join("../data/left", "*.jpg"))]
imgs = load_images(imgs_path)

### **Tarea A.2:** Detecte las esquinas de los patrones usando ``cv2.findChessboardCorners()``. Refine las detecciones con ``cv2.cornerSubPix()``.

In [71]:
# TODO Find corners with cv2.findChessboardCorners()
# Hint: you can use a list comprehension to process all images
corners = [cv2.findChessboardCorners(img, (8, 6)) for img in imgs]

In [72]:
corners_copy = copy.deepcopy(corners)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)

# TODO To refine corner detections with cv2.cornerSubPix() you need to input grayscale images. Build a list containing grayscale images.
imgs_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs]

corners_refined = [cv2.cornerSubPix(img_grey, corner[1], (5, 5), (-1, -1), criteria) if corner[0] else [] for img_grey, corner in zip(imgs_gray, corners_copy)]
print(corners_refined)

[[], array([[[109.26051 ,  95.83723 ]],

       [[127.9697  ,  92.68272 ]],

       [[145.5097  ,  89.7799  ]],

       [[161.68794 ,  87.1325  ]],

       [[176.8577  ,  84.6378  ]],

       [[190.81023 ,  82.46976 ]],

       [[203.91159 ,  80.506004]],

       [[216.19113 ,  78.53705 ]],

       [[123.14156 , 111.37146 ]],

       [[141.85524 , 107.610596]],

       [[159.32712 , 104.04907 ]],

       [[175.48586 , 100.694595]],

       [[190.47067 ,  97.756454]],

       [[204.46745 ,  95.07452 ]],

       [[217.32318 ,  92.48011 ]],

       [[229.42252 ,  90.23624 ]],

       [[137.82654 , 127.79829 ]],

       [[156.53664 , 123.30949 ]],

       [[173.87448 , 119.179565]],

       [[189.77519 , 115.34623 ]],

       [[204.67462 , 111.63686 ]],

       [[218.36482 , 108.317856]],

       [[231.1714  , 105.24443 ]],

       [[242.88799 , 102.44449 ]],

       [[153.45929 , 145.25757 ]],

       [[172.08199 , 139.89592 ]],

       [[189.15144 , 134.83945 ]],

       [[204.867   , 13

### **Tarea A.3:** Compruebe que las detecciones son correctas dibujando los resultados con ``cv2.drawChessboardCorners()``

In [73]:
imgs_copy = copy.deepcopy(imgs)

In [74]:
# TODO Use cv2.drawChessboardCorners() to draw the cornes
imgs_copy = [cv2.drawChessboardCorners(img, (8, 6), corner[1], True) for img, corner in zip(imgs_copy, corners) if corner[0]]

In [75]:
# TODO Show images and save when needed

def show_image(img):
    cv2.imshow(img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def write_image(img, filename):
    cv2.imwrite(filename, img)


### **Tarea A.4:** Defina y ejecute el método ``get_chessboard_points(chessboard_shape, dx, dy)``

In [76]:
# TODO Design the method. It should return a np.array with np.float32 elements
def get_chessboard_points(chessboard_shape, dx, dy):
    rows, cols = chessboard_shape
    points = []
    for i in range(cols):
        for j in range(rows):
            points.append([j * dx, i * dy, 0.0])
    return np.array(points, dtype=np.float32)

### **Tarea A.5:** Utilice ``cv2.calibrateCamera()`` para obtener los parámetros de calibración para la cámara izquierda

In [77]:
# Filter data and get only those with adequate detections
valid_corners = [corner[1] for corner in corners if corner[0]]
# Convert list to numpy array with np.float32 elements
valid_corners = np.asarray(valid_corners, dtype=np.float32)

In [78]:
# TODO You need the points for every image, not just one (consider a list comprehension)
chessboard_points = [get_chessboard_points((8, 6), 30.0, 30.0) for _ in range(len(valid_corners))]

chessboard_points = np.array(chessboard_points, dtype=np.float32)

In [79]:
# TODO
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(chessboard_points, valid_corners, (imgs[0].shape[1], imgs[0].shape[0]), None, None)

# Obtain extrinsics
extrinsics = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs, tvecs))

In [80]:
# Print outputs
print("Intrinsics:\n", intrinsics)
print("Distortion coefficients:\n", dist_coeffs)
print("Root mean squared reprojection error:\n", rms)

Intrinsics:
 [[419.17367964   0.         149.87203907]
 [  0.         420.67532299 128.88566155]
 [  0.           0.           1.        ]]
Distortion coefficients:
 [[ 2.54768626e-02 -3.34669238e+00  4.93017405e-03 -6.84826718e-03
   2.00281254e+01]]
Root mean squared reprojection error:
 0.15307311371562507


### **Pregunta A.1:** Repita el proceso (carga de imágenes, detección y comprobación de esquinas, etc.) para la cámara derecha.

In [81]:
# TODO Homework
imgs_path = [filename for filename in glob.glob(os.path.join("../data/right", "*.jpg"))]
imgs = load_images(imgs_path)

In [82]:
corners = [cv2.findChessboardCorners(img, (8, 6)) for img in imgs]

In [83]:
corners_copy = copy.deepcopy(corners)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)

# TODO To refine corner detections with cv2.cornerSubPix() you need to input grayscale images. Build a list containing grayscale images.
imgs_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs]

corners_refined = [cv2.cornerSubPix(img_grey, corner[1], (5, 5), (-1, -1), criteria) if corner[0] else [] for img_grey, corner in zip(imgs_gray, corners_copy)]
print(corners_refined)

[array([[[ 71.60907 , 107.567314]],

       [[ 91.33762 , 104.4793  ]],

       [[109.47375 , 101.462845]],

       [[126.63046 ,  98.68947 ]],

       [[142.55084 ,  96.30221 ]],

       [[157.32793 ,  94.02802 ]],

       [[171.06908 ,  91.94011 ]],

       [[184.17044 ,  90.15916 ]],

       [[ 85.09137 , 122.384094]],

       [[104.64154 , 118.586105]],

       [[123.06801 , 115.11728 ]],

       [[140.08055 , 111.69478 ]],

       [[155.86206 , 108.7475  ]],

       [[170.56097 , 105.989334]],

       [[184.3586  , 103.488815]],

       [[197.19096 , 101.17556 ]],

       [[ 99.29753 , 138.24942 ]],

       [[118.95734 , 133.63411 ]],

       [[137.26688 , 129.47122 ]],

       [[154.15654 , 125.65168 ]],

       [[169.83392 , 122.096794]],

       [[184.4291  , 118.72657 ]],

       [[198.16696 , 115.635635]],

       [[210.66168 , 112.8909  ]],

       [[114.27154 , 154.73987 ]],

       [[133.97429 , 149.48792 ]],

       [[152.09679 , 144.57715 ]],

       [[168.80356 , 139.98

In [84]:
imgs_copy_right = copy.deepcopy(imgs)

In [85]:
imgs_copy = [cv2.drawChessboardCorners(img, (8, 6), corner[1], True) for img, corner in zip(imgs_copy, corners) if corner[0]]

In [86]:
# Filter data and get only those with adequate detections
valid_corners = [corner[1] for corner in corners if corner[0]]
# Convert list to numpy array with np.float32 elements
valid_corners = np.asarray(valid_corners, dtype=np.float32)

In [87]:
chessboard_points = [get_chessboard_points((8, 6), 30.0, 30.0) for _ in range(len(valid_corners))]

chessboard_points = np.array(chessboard_points, dtype=np.float32)

In [88]:
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(chessboard_points, valid_corners, (imgs[0].shape[1], imgs[0].shape[0]), None, None)

# Obtain extrinsics
extrinsics = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs, tvecs))

In [89]:
# Print outputs
print("Intrinsics:\n", intrinsics)
print("Distortion coefficients:\n", dist_coeffs)
print("Root mean squared reprojection error:\n", rms)

Intrinsics:
 [[432.19183608   0.         146.77820873]
 [  0.         435.44775772 137.27781915]
 [  0.           0.           1.        ]]
Distortion coefficients:
 [[-0.1186237  -0.186011    0.00625041 -0.00822418  1.08945459]]
Root mean squared reprojection error:
 0.205888506208714


### **Pregunta A.2:** Diferencias entre cv2.findChessboardCorners() y cv2.cornerSubPix()

In [90]:
# TODO Homework

### **Pregunta A.3:** Número mínimo de imágenes necesarias para calibrar.

In [91]:
# TODO Homework

## **Apartado B: Corrección de distorsión** (ojo de pez)

En este apartado se trabajará en la corrección de la distorsión debido a lentes de ojo de pez. Primero se calibrará una cámara con este tipo de lente, utilizando las imágenes de la carpeta ``fisheye``. Posteriormente se utilizarán estos parámetros de calibración para corregir la distorsión de una de las imágenes de calibración.

Los pasos que deberá seguir para calibrar una cámara con distorsión de ojo de pez son:

1. Reutilice el método ``load_images()`` para cargar las imágenes de la carpeta ``fisheye``.
2. Detecte las equinas procesando las imágenes con los métodos ``cv2.findChessboardCorners()`` y ``cv2.cornerSubPix()``.
3. Reutilice la función ``get_chessboard_points()`` para obtener las coordenadas del tablero.
4. Defina los argumentos para la función de calibración.
5. Calibre con ``cv2.fisheye.calibrate()``

### **Tarea B.1:** Reutilice el método ``load_images()`` para cargar las imágenes de la carpeta ``fisheye``

In [92]:
# TODO Build a list containing the paths of all images from the fisheye camera and load images
fisheye_imgs_path = []
fisheye_imgs = load_images()

TypeError: load_images() missing 1 required positional argument: 'filenames'

### **Tarea B.2:** Detecte las equinas procesando las imágenes con los métodos ``cv2.findChessboardCorners()`` y ``cv2.cornerSubPix()``.

In [ ]:
imgs_corners = []
# Parameters for cv2.cornerSubPix()
subpix_criteria = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 30, 0.1)

#TODO Complete the required parts of the loop
for img in fisheye_imgs:
    
    # TODO parse arguments to cv2.findChessboardCorners()
    corners = cv2.findChessboardCorners()
    
    # TODO convert image to grayscale to use cv2.cornerSubPix()
    gray_img = cv2.cvtColor()
    refined_corners = cv2.cornerSubPix(gray_img, corners, (3,3), (-1,-1), subpix_criteria)

    # TODO append only those refined_corners with proper detections
    imgs_corners.append(refined_corners)

### **Tarea B.3:** Reutilice la función ``get_chessboard_points()`` para obtener las coordenadas del tablero

In [ ]:
# TODO Define the chessboard dimensions and the lenght of the squares (in [mm])
chessboard_dims = (None, None)
length = None
# TODO You need the points for every image, not just one (consider a list comprehension)
fisheye_chessboard_points = get_chessboard_points(chessboard_dims, length, length)

### **Tarea B.4:** Defina los argumentos para la calibración

In [ ]:
# Parameters for cv2.fisheye.calibrate()
calibration_flags = cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC+cv2.fisheye.CALIB_FIX_SKEW
intrinsics = np.zeros((3, 3))
distortion = np.zeros((4, 1))
rotations = [np.zeros((1, 1, 3), dtype=np.float64) for _ in imgs_corners]
traslations = [np.zeros((1, 1, 3), dtype=np.float64) for _ in imgs_corners]


### **Tarea B.5:** Calibración

In [ ]:
rms, _, _, _, _ = \
cv2.fisheye.calibrate(fisheye_chessboard_points, imgs_corners, gray_img.shape[::-1], intrinsics, distortion, rotations, traslations, calibration_flags, subpix_criteria)

In [ ]:
# Show intrinsic matrix and distortion coefficients values
print(intrinsics)
print(distortion)
print(rms)

### **Pregunta B.1:** Corrija la distorsión de las 2 primeras imágenes de la carpeta ``fisheye``

In [ ]:
# TODO Search in the documentation to define 'dim'
dim = None
map1, map2 = cv2.fisheye.initUndistortRectifyMap(intrinsics, distortion, np.eye(3), intrinsics, dim, cv2.CV_16SC2)

In [ ]:
# TODO Homework: correct distortion using cv2.remap()